# Master Thesis Martin Vingerhoets 

## Section 1: Introduction 

### Staggered Grid Spatial Discretization of 1D and 2D SWE  

Q1: How to set the mesh width (i.e. the spatial resolution) in a non-linear transient problem in case that: 
1. higher harmonics are generated (by the non-linesar terms in the equation) and;
2. one wishes to preserve a minimal number of grid point per wave-length?

Q2: what are reference implementations of staggered grid methods 
1. see [staggered grid](https://ptsolvers.github.io/JustRelax.jl/dev/man/equations_discretization) in JustRelax.jl; 

### Explicit, Implicit and Hybrid Transient Solver 

Q1: How should a [steady state solver](https://docs.sciml.ai/DiffEqDocs/stable/types/steady_state_types/) be applied to solve a transient problem? Test this first on the harmonic oscillator. 

Q2: What kind of IMEX solvers ([Split ODE Problems](https://docs.sciml.ai/DiffEqDocs/stable/types/split_ode_types/)) are available to simulate shallow water flow? What are the implicit and explicit solvers that are being combined? 

Q3: How should IMEX solvers be deployed such that all non-linear terms are treated explicitly and that merely a **linear** system needs to be solver at each time? 

Q4: How are the linear systems resulting from IMEX methods solved? Can the resulting linear system be solved using a Schur (or approximation thereof) complement method as e.g. in [shallowWaterFoam.C](https://www.openfoam.com/documentation/guides/latest/api/shallowWaterFoam_8C.html#details). 

### (Non-)Linear Time-Harmonic Solver 

Decompose time-dependent part into harmonics (base harmonic and higher order harmonic generated by the non-linear term in the equation). Hpowq

Q1: what non-linear terms are taken into account? How is the term $\| \mathbf{u} \| \mathbf{u}$ treated? How can this term be simplified without loosing too much of the physics involved?  

Q2: how can the sparse Jacobian be generated using [ForwardDiff](https://github.com/JuliaDiff/ForwardDiff.jl). How to use [sparsity dedection](https://docs.sciml.ai/NonlinearSolve/stable/tutorials/large_systems/#Approximate-Sparsity-Detection-and-Sparse-Jacobians)? 

Q3: what is the structure of the Jacobian? How to split Jacobian into a linear part (independent of the Newton step) and an non-linear part (varrying with the  Newton step);

Q4: how to apply a conformal transformation on the linear part of the Jacobian to obtain all eigenvalues on the same side of the imaginary axis? 

Q5: how to freeze the preconditioner over various iterations on the Newton iteration? 

### Newton-LU for both Transient and Time-Harmonic Solvers

<b>Advantages of LU:</b> Robust. Mainstream, and thus well documented. What about modern variatiants (e.g. Pardiso), recursive factorizations or GPU implementations?    

<b>Disadvantages of LU:</b>

1. the large number of fill-in elements, especially in case of two-dimensional discretization with large number of harmonics;
2. solving too accuratey (oversolving) especially at first Newton iterations, when still far from the final solution (cfr. forcing terms, sufficient descent directions);

### Newton-Krylov for both Transient and Time-Harmonic Solvers

<b>Advantages of Krylov:</b> Avoid oversolving using [forcing_strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies). 

<b>Disadvantages of Krylov:</b> Slow to converge in case that the eigenvalues of the Jacobian spread accross both sides of the imaginary axis unless properly treated (using a conformal mapping as in the case of the complex-shifted Laplace preconditioner). 

Q1: How does one ensure that the GMRES search space is allocated once only, accross all of the Newton iterations? LinearSolve.jl builds it once in its cache init and then future solves just reinit!. 

Q2: how to specify the preconditioner in the example on the page [forcing strategies](https://docs.sciml.ai/NonlinearSolve/stable/native/solvers/#forcing_strategies).

Q3: what does <i>reuse_A_if_factorization</i> refer to? 

References: book by Tim Kelley on Non-Linear Equations, papers by e.g. Dembo-Eisenstat; 

### Packages for Krylov Subspace Solvers 

1. [KrylovKit.jl](https://jutho.github.io/KrylovKit.jl/stable/man/linear/)
2. [Krylov.jl](https://jso.dev/Krylov.jl/stable/) In particular [restarted GMRES](https://jso.dev/Krylov.jl/stable/solvers/unsymmetric/#GMRES)
3. [iterativesolvers](https://iterativesolvers.julialinearalgebra.org/dev/)
4. preconditioners [BlockArrays](https://github.com/JuliaArrays/BlockArrays.jl)

### GMRES Solvers 

1. restart value?
2. initial guess?
3. stopping criterium? 

### Scalar (pointwise) Preconditioners for Krylov

How does ILU out of the box perform? Can the ILU decompositon of the Jacobian froozen over various Newton iterations. How many GMRES iterations can be solved by implementing forcing terms?  

### Exploiting the Block Structure of the Jacobian in Preconditioning   

Assume a spatial discretization using $N_c$ cells (or elements or any alternative denoting the spatial accuracy) and using $N_h$ harmonics. Then the Jacobian is of size $N_c*N_h$. Martin showed the Jacobian as $N_c$ blocks, of size $N_h$. We instead would like to see the Jacobian as $N_h$ blocks of size $N_d$ (thus arriving at a smaller number of blocks that are large in size, especially in case of two-dimensional discretizations at high spatial resolution). 

### Newton-Krylov-BlockJacobi 

Can GMRES at each Newton iteration be precondition using block Jacobi? 

### Parallel in Frequency using pmap 

Can the $N_h$ blocks of size $N_c$ be solved independently from each other using the function <i>pmap</i>? 

### Spatial Coarse Grid to Curb Increase in GMRES Iterations as $N_h$ Increases  

Can the increase in GMRES iterationss with $N_h$ bew curbed using coarser grids (lower spatial resolution).

## Section 2: 1D Shallow Water 

### Model Equations 

### Transient Analysis 

### Time-Harmonic Analysis 

## Section 3: 2D Shallow Water 

### Transient Analysis 

### Time-Harmonic Analysis 

## Section 4: Bifurcation Analysis 

Solve for gradually smaller height of the channel (as the non-linearity is proportional to inverse of height).  

## Section 5: Conclusions 

## References 

Hierbij de link naar de files met het model:

https://filesender.surf.nl/?s=download&token=a7bd1605-4e56-485e-96e3-6d1750ba2572

Ik heb nog even gevraagd hoe de auteur (Haoyan) de niet-lineaire term doet. Hij doet inderdaad wat Martin voorstelde:

Mijn vraag: Do you substitute the velocity signal from the previous iteration and then do a FT, or do you have a 'closed' form solution?

Antwoord:

I followed the first approach: substituting the velocity signal from the previous iteration and then applying the Fourier transform.

As shown in Equation S.14, \phi^{j-1} indicates that it comes from the previous iteration.

Misschien kan dit slimmer (Domenico weet misschien een truc), maar deze werkt iig. Weet niet of het slim is om te discretiseren, ik denk dat het beter is een functie te maken (met de coefficienten uit de vorige iteratie), die te vermenigvuldigen met een specifieke mode (gebruik orthogonaliteit) en die dan met een slimme integrator te evalueren. Kunnen we het volgende week over hebben.

Merk trouwens ook op dat Haoyan niet vermenigvuldigd met (H+\zeta-h) maar die samen met de |u| numeriek evalueert zoals boven geschreven.